# OSL Baseline — Bilateral-sensor Chemotaxis


In [ ]:
import os, sys, subprocess

if os.path.isdir('/content'):
    REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
    REPO_DIR = '/content/2d-osl'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    REPO_DIR = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_DIR, 'src')) and REPO_DIR != os.path.dirname(REPO_DIR):
        REPO_DIR = os.path.dirname(REPO_DIR)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('repo:', REPO_DIR, '\ncwd :', os.getcwd())


## Smoke check


In [ ]:
import numpy as np
from src.envs.osl_env import EnvConfig, OslEnv
from src.baselines.chemotaxis import BilateralChemotaxis, ChemotaxisConfig, run_episode

env = OslEnv()
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'action_space', env.action_space.shape)

ctrl = BilateralChemotaxis()
a = ctrl.act(obs)
print('action', a, '(continuous [v, body_omega, head_omega] in [-1, 1])')
r = run_episode(env, ctrl, seed=0)
print('one clean episode →', {k: r[k] for k in ('return', 'success', 'casts', 'steps')})

## Configuration


In [ ]:
ENV_KW = dict(
    sensor_spacing_mm=0.15,
    episode_seconds=120.0,
    arena_width_mm=80.0, arena_height_mm=120.0,
    source_x_mm=40.0, source_y_mm=100.0,
    gaussian_sigma_mm=30.0, success_radius_mm=7.5,
)

CTRL_KW = dict(
    steer_gain=80.0,
    surge_speed=1.0, cautious_speed=0.2,
    ewma_alpha=0.3, weak_frac=0.35,
    rising_deadband_frac=0.05,
    cast_after_weak_steps=4,
    cast_head_omega=1.0, cast_half_period=6,
    cast_creep_speed=0.05, cast_body_omega=0.25,
)

EVAL_SEED_BASE = 20_000
EVAL_EPISODES = 100
EVAL_NOISE_STAGE = 0
EVAL_NOISE_STRENGTH = 0.0

import numpy as np
from src.envs.osl_env import EnvConfig, OslEnv
from src.baselines.chemotaxis import BilateralChemotaxis, ChemotaxisConfig, run_episode

def make_env(stage, strength, seed):
    cfg = {**ENV_KW, 'noise_stage': int(stage), 'noise_strength': float(strength), 'seed': seed}
    return OslEnv(EnvConfig.from_dict(cfg))

controller = BilateralChemotaxis(ChemotaxisConfig(**CTRL_KW))
print('controller ready:', CTRL_KW)


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.titleweight': 'normal',
    'axes.labelsize': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': 150,
})

ACCENT = 'steelblue'


## Evaluation


In [ ]:
import numpy as np

def evaluate(stage, strength, n_episodes=EVAL_EPISODES, seed_base=EVAL_SEED_BASE):
    succ, steps, rets, casts = [], [], [], []
    for i in range(n_episodes):
        seed = seed_base + i
        env = make_env(stage, strength, seed)
        r = run_episode(env, controller, seed=seed)
        succ.append(int(r['success'])); steps.append(r['steps'])
        rets.append(r['return']); casts.append(r['casts'])
    succ = np.asarray(succ); steps = np.asarray(steps); casts = np.asarray(casts)
    succ_steps = steps[succ == 1]
    cast_frac = casts / np.maximum(steps, 1)
    return {
        'stage': stage, 'strength': strength, 'n': n_episodes,
        'success_rate': float(succ.mean()),
        'mean_steps_all': float(steps.mean()),
        'mean_steps_success': float(succ_steps.mean()) if len(succ_steps) else float('nan'),
        'mean_return': float(np.mean(rets)),
        'mean_casts': float(casts.mean()),
        'cast_fraction': float(cast_frac.mean()),
    }

clean = evaluate(0, 0.0)
print(f"CLEAN field (stage 0): success={clean['success_rate']:.0%}  "
      f"mean_steps(success)={clean['mean_steps_success']:.0f}  mean_return={clean['mean_return']:.2f}  "
      f"casts={clean['mean_casts']:.1f}  cast_frac={clean['cast_fraction']:.1%}")
assert clean['success_rate'] == 1.0, 'Clean field should be solved on every episode!'
print('baseline solves the clean field on every episode.')


## Trajectory PNG


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import Image as DisplayImage, display
except ImportError:
    DisplayImage = None
    def display(_):
        pass
from src.utils.plotter import _plume_field

TRAJ_STAGE = 0
TRAJ_STRENGTH = 0.0
TRAJ_N_SEEDS = 10
TRAJ_SEEDS = None            # None = auto-pick successful seeds; or set a list like [20000, 20001]

selected_seeds = list(TRAJ_SEEDS) if TRAJ_SEEDS is not None else []
if not selected_seeds:
    for i in range(500):
        s = EVAL_SEED_BASE + i
        env = make_env(TRAJ_STAGE, TRAJ_STRENGTH, s)
        rr = run_episode(env, controller, seed=s)
        if rr['success']:
            selected_seeds.append(s)
        if len(selected_seeds) >= TRAJ_N_SEEDS:
            break
if not selected_seeds:
    selected_seeds = [EVAL_SEED_BASE + i for i in range(TRAJ_N_SEEDS)]
print('plotting seeds', selected_seeds)

def rollout_trace(stage, strength, seed):
    env = make_env(stage, strength, seed)
    obs, _ = env.reset(seed=seed)
    controller.reset()
    ret, casts, success = 0.0, 0, False
    traj_x, traj_y, active_x, active_y = [], [], [], []
    for t in range(env.max_steps):
        traj_x.append(env.x_mm); traj_y.append(env.y_mm)
        action = controller.act(obs)
        obs, r, term, trunc, info = env.step(action)
        ret += float(r)
        if info.get('event_is_high_cast_like'):
            casts += 1
            active_x.append(env.x_mm); active_y.append(env.y_mm)
        if term or trunc:
            success = bool(info.get('success', False))
            break
    return dict(
        seed=seed, return_=ret, success=success, casts=casts, steps=t + 1, env=env,
        traj_x=traj_x, traj_y=traj_y, active_x=active_x, active_y=active_y,
    )

rollouts = [rollout_trace(TRAJ_STAGE, TRAJ_STRENGTH, seed) for seed in selected_seeds]
print(f"success={sum(r['success'] for r in rollouts)}/{len(rollouts)} "
      f"mean_steps={np.mean([r['steps'] for r in rollouts]):.1f} "
      f"mean_active_sensing={np.mean([r['casts'] for r in rollouts]):.1f}")

os.makedirs('runs/baseline_chemotaxis/plots', exist_ok=True)
png_path = (f'runs/baseline_chemotaxis/plots/'
            f'baseline_trajectories_n{len(rollouts)}_stage{TRAJ_STAGE}_alpha{TRAJ_STRENGTH}.png')

field, W, H = _plume_field(rollouts[0]['env'])
cfg = rollouts[0]['env'].cfg
fig, ax = plt.subplots(figsize=(5.2, 7.6))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
ax.imshow(field, extent=[0.0, W, 0.0, H], origin='lower', cmap='magma',
          vmin=0.0, vmax=max(1e-6, float(field.max()) * 1.2))
colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(10, len(rollouts))))
for idx, result in enumerate(rollouts):
    color = colors[idx % len(colors)]
    ax.plot(result['traj_x'], result['traj_y'], color=color, linewidth=2.0, alpha=0.86)
    if result['active_x']:
        ax.scatter(result['active_x'], result['active_y'], color='white', marker='*', s=48,
                   edgecolors='black', linewidths=0.35, alpha=0.75, zorder=10)
    ax.scatter([result['traj_x'][0]], [result['traj_y'][0]], color='white', marker='o', s=42,
               edgecolors='black', linewidths=0.45, alpha=0.9, zorder=11)
    ax.scatter([result['traj_x'][-1]], [result['traj_y'][-1]], color=color, marker='X', s=68,
               edgecolors='black', linewidths=0.45, zorder=11)
ax.scatter([cfg.source_x_mm], [cfg.source_y_mm], color='lime', marker='P', s=150,
           edgecolors='black', linewidths=0.6, zorder=12)
ax.add_patch(plt.Circle((cfg.source_x_mm, cfg.source_y_mm), cfg.success_radius_mm,
                        color='white', fill=False, linewidth=1.2, alpha=0.85))
ax.set_xlim(0.0, W); ax.set_ylim(0.0, H)
ax.set_xlabel('x (mm)', color='white'); ax.set_ylabel('y (mm)', color='white')
ax.set_aspect('equal')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')
fig.tight_layout()
fig.savefig(png_path, dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print('[PNG] Saved to', png_path)
if DisplayImage is not None:
    display(DisplayImage(filename=png_path))


## Noise sweep


In [ ]:
import json
import matplotlib.pyplot as plt

SWEEP = [
    (0, 0.0), (1, 0.3), (1, 0.6), (1, 1.0),
    (2, 0.3), (2, 0.6), (2, 1.0),
]
SWEEP_EPISODES = 60   # per condition (lower than EVAL_EPISODES for speed)

rows = [evaluate(stage, strength, n_episodes=SWEEP_EPISODES) for stage, strength in SWEEP]

print(f"{'stage':>5} {'α':>4} {'success':>8} {'steps(succ)':>12} {'return':>8} {'casts':>7} {'cast%':>7}")
for r in rows:
    print(f"{r['stage']:>5} {r['strength']:>4.1f} {r['success_rate']:>7.0%} "
          f"{r['mean_steps_success']:>12.0f} {r['mean_return']:>8.2f} "
          f"{r['mean_casts']:>7.1f} {r['cast_fraction']:>6.1%}")

labels = [f"s{r['stage']}·α{r['strength']}" for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].bar(labels, [r['success_rate'] for r in rows], color=ACCENT)
ax[0].set_ylim(0, 1); ax[0].set_ylabel('success rate')
ax[0].set_title('Success ratio')
ax[0].tick_params(axis='x', rotation=30)
ax[1].bar(labels, [r['mean_steps_success'] for r in rows], color=ACCENT)
ax[1].set_ylabel('steps to source')
ax[1].set_title('Steps to source')
ax[1].tick_params(axis='x', rotation=30)
ax[2].bar(labels, [100.0 * r['cast_fraction'] for r in rows], color=ACCENT)
ax[2].set_ylabel('cast steps (%)')
ax[2].set_title('Cast fraction')
ax[2].tick_params(axis='x', rotation=30)
fig.tight_layout()
os.makedirs('runs/baseline_chemotaxis', exist_ok=True)
fig.savefig('runs/baseline_chemotaxis/noise_sweep.png', dpi=150)
with open('runs/baseline_chemotaxis/noise_sweep.json', 'w') as f:
    json.dump(rows, f, indent=2)
plt.show()
print('\n[saved] runs/baseline_chemotaxis/noise_sweep.{png,json}')
